In [1]:
# Standard libraries
from pathlib import Path
import os
import random
import copy

# Numerical computing
import numpy as np
import pandas as pd

# Image handling
from PIL import Image

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Torchvision
from torchvision import models, transforms

# Metrics
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Visualization (for t-SNE)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE

# Progress bars
from tqdm import tqdm

In [2]:
import torch
import torch.nn as nn
from torchvision import models

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [4]:
model = models.efficientnet_b0(weights=None)

model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(1280, 512),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(512, 3)
)

In [5]:
transform = (
    models.EfficientNet_B0_Weights
    .DEFAULT
    .transforms()
)

In [6]:
feature_extractor = nn.Sequential(
    model.features,
    model.avgpool,
    nn.Flatten(),
    model.classifier[0],
    model.classifier[1],
    model.classifier[2]
).to(device)

feature_extractor.eval()

Sequential(
  (0): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv

In [7]:
x = torch.randn(1, 3, 224, 224).to(device)

with torch.no_grad():
    emb = feature_extractor(x)

print(emb.shape)

torch.Size([1, 512])


In [8]:
PROJECT_ROOT = Path.cwd().parent

EMBED_ROOT = (
    PROJECT_ROOT
    / "processed"
    / "UTA"
    / "embeddings"
)

NORM_ROOT = (
    PROJECT_ROOT
    / "processed"
    / "UTA"
    / "normalized_embeddings"
)

NORM_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

In [9]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

UTA_ROOT = (
    PROJECT_ROOT
    / "processed"
    / "UTA"
    / "sequences"
)

In [10]:
subjects = sorted(
    [p.name for p in EMBED_ROOT.iterdir()]
)

# Remove incomplete subject
subjects = [s for s in subjects if s != "46"]

for subject in tqdm(subjects):

    alert_path = EMBED_ROOT / subject / "alert.npy"
    low_path = EMBED_ROOT / subject / "low_vigilant.npy"
    drowsy_path = EMBED_ROOT / subject / "drowsy.npy"

    # Skip if any file is missing
    if not (
        alert_path.exists()
        and low_path.exists()
        and drowsy_path.exists()
    ):
        print(f"Skipping {subject}: missing file")
        continue

    alert = np.load(alert_path)
    low = np.load(low_path)
    drowsy = np.load(drowsy_path)

    # Skip if any embedding array is empty
    if (
        len(alert) == 0
        or len(low) == 0
        or len(drowsy) == 0
    ):
        print(
            f"Skipping {subject}: "
            f"{alert.shape}, {low.shape}, {drowsy.shape}"
        )
        continue

    save_dir = NORM_ROOT / subject
    save_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # Compute baseline statistics from alert state
    mean = alert.mean(axis=0)
    std = alert.std(axis=0)

    std = np.clip(std, a_min=1e-2, a_max=None)

    # Normalize
    alert_norm = (alert - mean) / std
    low_norm = (low - mean) / std
    drowsy_norm = (drowsy - mean) / std

    # Save normalized embeddings
    np.save(
        save_dir / "alert.npy",
        alert_norm.astype(np.float32)
    )

    np.save(
        save_dir / "low_vigilant.npy",
        low_norm.astype(np.float32)
    )

    np.save(
        save_dir / "drowsy.npy",
        drowsy_norm.astype(np.float32)
    )

    # Save normalization parameters
    np.save(
        save_dir / "mean.npy",
        mean.astype(np.float32)
    )

    np.save(
        save_dir / "std.npy",
        std.astype(np.float32)
    )

print("Normalization complete.")

  0%|          | 0/47 [00:00<?, ?it/s]

 17%|█▋        | 8/47 [00:00<00:00, 73.01it/s]

 34%|███▍      | 16/47 [00:00<00:00, 71.93it/s]

 51%|█████     | 24/47 [00:00<00:00, 64.86it/s]

 66%|██████▌   | 31/47 [00:00<00:00, 65.70it/s]

 81%|████████  | 38/47 [00:00<00:00, 65.01it/s]

 96%|█████████▌| 45/47 [00:00<00:00, 65.31it/s]

100%|██████████| 47/47 [00:00<00:00, 65.07it/s]

Normalization complete.


In [11]:
x = np.load(
    NORM_ROOT / "01" / "alert.npy"
)

print(x.mean())
print(x.std())

-6.1231096e-09
0.80921954


In [12]:
print(x[:, 0].mean())
print(x[:, 0].std())

-1.959395e-07
0.9999997


In [13]:
frame_paths = []

for folder in ["10_1", "10_2"]:
    p = UTA_ROOT / "32" / folder

    if p.exists():
        frame_paths.extend(
            sorted(p.glob("*.jpg"))
        )

print("Total frames:", len(frame_paths))

Total frames: 595


In [14]:
from PIL import Image
from tqdm import tqdm
import numpy as np

embeddings = []

for img_path in tqdm(frame_paths):

    img = Image.open(img_path).convert("RGB")

    x = (
        transform(img)
        .unsqueeze(0)
        .to(device)
    )

    with torch.no_grad():
        emb = feature_extractor(x)

    embeddings.append(
        emb.squeeze().cpu().numpy()
    )

embeddings = np.array(
    embeddings,
    dtype=np.float32
)

print(embeddings.shape)

np.save(
    EMBED_ROOT / "32" / "drowsy.npy",
    embeddings
)

  0%|          | 0/595 [00:00<?, ?it/s]

  1%|          | 3/595 [00:00<00:23, 25.66it/s]

  2%|▏         | 10/595 [00:00<00:11, 49.97it/s]

  3%|▎         | 17/595 [00:00<00:09, 58.28it/s]

  4%|▍         | 24/595 [00:00<00:09, 60.94it/s]

...

 97%|█████████▋| 579/595 [00:09<00:00, 59.42it/s]

 98%|█████████▊| 585/595 [00:09<00:00, 58.30it/s]

 99%|█████████▉| 591/595 [00:09<00:00, 57.17it/s]

100%|██████████| 595/595 [00:09<00:00, 60.79it/s]

(595, 512)


In [15]:
print(np.load(
    EMBED_ROOT / "32" / "drowsy.npy"
).shape)

(595, 512)


In [16]:
subject = "32"

save_dir = NORM_ROOT / subject
save_dir.mkdir(parents=True, exist_ok=True)

alert = np.load(EMBED_ROOT / subject / "alert.npy")
low = np.load(EMBED_ROOT / subject / "low_vigilant.npy")
drowsy = np.load(EMBED_ROOT / subject / "drowsy.npy")

mean = alert.mean(axis=0)
std = alert.std(axis=0)
std = np.clip(std, a_min=1e-2, a_max=None)

alert_norm = (alert - mean) / std
low_norm = (low - mean) / std
drowsy_norm = (drowsy - mean) / std

np.save(save_dir / "alert.npy", alert_norm.astype(np.float32))
np.save(save_dir / "low_vigilant.npy", low_norm.astype(np.float32))
np.save(save_dir / "drowsy.npy", drowsy_norm.astype(np.float32))
np.save(save_dir / "mean.npy", mean.astype(np.float32))
np.save(save_dir / "std.npy", std.astype(np.float32))

print("Subject 32 normalized successfully.")

Subject 32 normalized successfully.


In [17]:
print(np.load(NORM_ROOT / "32" / "drowsy.npy").shape)

(595, 512)
